# `optical_correction()` — `chavez-cost-1996`

Aplicar correção óptica COST a uma banda em reflectância TOA.

**Cadeia deste exemplo:** Reflectância TOA + QA + elevação solar → ajuste COST → reflectância corrigida.

Este notebook executa uma etapa sobre arquivos já preparados. **Se você tem apenas os arquivos originais Level-1, prepare as entradas indicadas na tabela antes de executar este método.**

Execute apenas os **quatro blocos de código**, de cima para baixo. Os blocos ao final, em “Consulta”, são alternativas para leitura, e não uma continuação obrigatória.


## Antes de executar: separe estes arquivos

| Campo a preencher no passo 1 | Arquivo que você deve escolher | Unidade / tipo | Como obter se ainda falta |
| --- | --- | --- | --- |
| `rho_toa_path` | Banda óptica a corrigir; por exemplo, `reflectancia_TOA_B4.tif` | Reflectância TOA, adimensional, já ajustada pelo ângulo solar | [Reflectância TOA](toa_reflectance.ipynb) para a banda óptica desejada |
| `qa_pixel_path` | Qualidade dos pixels; por exemplo, `PRODUTO_QA_PIXEL.TIF` | Bits inteiros, sem calibração | Arquivo original QA_PIXEL da mesma cena e na mesma grade |
| `qa_radsat_path` | Saturação/qualidade geométrica; por exemplo, `PRODUTO_QA_RADSAT.TIF` | Bits inteiros, sem calibração | Arquivo original QA_RADSAT da mesma cena e na mesma grade |

Todos os GeoTIFFs devem representar a mesma cena e ter dimensões, CRS e transformação idênticos. Para recortes, use a mesma janela nas QA. As unidades são verificadas pelo usuário: a leitura do arquivo não comprova sua grandeza física.

**Configuração desta execução:** Preencha `elevacao_solar` com SUN_ELEVATION do MTL em graus. Execute uma banda óptica por vez; escolha uma pasta de saída diferente para cada banda/cadeia. O percentil é ajustado depois da exclusão QA.

**Ambiente:** instale `python -m pip install -e ".[examples]"` na pasta da biblioteca e selecione o kernel desse ambiente no VS Code/Jupyter.


## Passo 1 — Preencher caminhos e escolher a saída

Substitua cada `CAMINHO_...` pelo caminho completo do seu arquivo, mantendo `Path(...)`. No Windows, escreva, por exemplo, `Path(r"C:\dados\radiancia_B4.tif")`. Preencha todos os campos da tabela, inclusive as duas QA.

Preencha `elevacao_solar` com SUN_ELEVATION do MTL em graus. Execute uma banda óptica por vez; escolha uma pasta de saída diferente para cada banda/cadeia. O percentil é ajustado depois da exclusão QA.

Ajuste `saida` para a pasta desejada. Caminhos relativos partem da pasta de trabalho do kernel; a célula imprime o caminho absoluto. Em nova execução com o mesmo destino, o GeoTIFF é substituído.


In [ ]:
from pathlib import Path
import numpy as np
import rasterio

# EDITE OS CAMINHOS E AS CONFIGURAÇÕES ABAIXO; mantenha o método deste notebook.
from pylstemp import quality_mask
from pylstemp import optical_correction

rho_toa_path = Path("CAMINHO_DA_REFLECTANCIA_TOA.tif")
qa_pixel_path = Path("CAMINHO_DO_QA_PIXEL.tif")
qa_radsat_path = Path("CAMINHO_DO_QA_RADSAT.tif")

elevacao_solar = None  # informar a elevação real da cena, em graus
if elevacao_solar is None:
    raise ValueError('Informe elevacao_solar em graus, usando o MTL da cena.')
diagnosticos = False
saida = Path("resultados/optical_correction__chavez-cost-1996")

# Confira os arquivos antes de passar ao passo 2.
entradas_obrigatorias = {
    "rho_toa_path": rho_toa_path,
    "qa_pixel_path": qa_pixel_path,
    "qa_radsat_path": qa_radsat_path,
}
pendentes = [f"{nome}: {caminho}" for nome, caminho in entradas_obrigatorias.items()
             if not caminho.is_file()]
if pendentes:
    raise FileNotFoundError("Preencha os caminhos de entrada com arquivos existentes:\n"
                            + "\n".join(pendentes))
print("Arquivos encontrados. Próximo passo: ler os dados e aplicar QA.")
print("Pasta de saída:", saida.resolve())


## Passo 2 — Ler as entradas, conferir a grade e aplicar QA

Execute esta célula depois de confirmar os caminhos. Ela carrega os dados, confere seu alinhamento, lê QA_PIXEL/QA_RADSAT com `quality_mask()` e combina QA com nodata em `mascara`. `True` indica pixel excluído. A leitura não recalibra dados prontos.

Água é preservada na política QA deste exemplo (`exclude_water=False`). A célula imprime as dimensões e as contagens de exclusão.


In [ ]:
arquivos = {
    "rho_toa": rho_toa_path,
}
dados = {}
grade = None
for nome, caminho in arquivos.items():
    with rasterio.open(caminho) as src:
        grade_atual = (src.shape, src.crs, src.transform)
        if grade is None:
            grade = grade_atual
            perfil = src.profile.copy()
        elif grade_atual != grade:
            raise ValueError(f"A imagem {nome} não está na mesma grade das outras entradas.")
        dados[nome] = src.read(1, masked=True).astype(float).filled(np.nan)
rho_toa = dados["rho_toa"]
mascara = ~np.isfinite(rho_toa)
# Ler QA na grade da primeira imagem e combinar com nodata antes do método.
mascara_qa = quality_mask(
    qa_pixel_path, qa_radsat_path,
    reference_path=next(iter(arquivos.values())),
    exclude_water=False,
)
mascara |= mascara_qa

print("Dimensões:", mascara.shape)
print("Pixels excluídos por QA:", int(mascara_qa.sum()))
print("Pixels excluídos por QA ou nodata:", int(mascara.sum()), "de", mascara.size)


## Passo 3 — Calcular o produto

Execute a chamada abaixo. As variáveis são matrizes carregadas no passo 2; os caminhos de arquivo foram usados no passo 1. `mask=mascara` aplica a seleção QA/nodata ao cálculo.


In [ ]:
resultado = optical_correction(
    method='chavez-cost-1996',
    toa_reflectance=rho_toa,
    sun_elevation=elevacao_solar,
    percentile=.01,
    assumed_dark_surface=.01,
    mask=mascara,
    return_diagnostics=diagnosticos,
)
imagem = resultado["reflectance"] if diagnosticos else resultado

## Passo 4 — Salvar e usar a saída

Execute esta célula para gravar o GeoTIFF em `saida`, preservando a grade. Confira o caminho absoluto e a quantidade de pixels válidos impressos. Pixels inválidos são gravados como NaN.

**Próximo uso:** Preparar vermelho/NIR corrigidos para cadeias que aceitam essa convenção óptica. O produto não substitui radiância TOA em métodos que exigem radiância.


In [ ]:
saida.mkdir(parents=True, exist_ok=True)
perfil.update(driver="GTiff", dtype="float32", count=1, nodata=np.nan, compress="deflate")
with rasterio.open(saida / "optical_correction__chavez-cost-1996.tif", "w", **perfil) as dst:
    dst.write(np.asarray(imagem, dtype="float32"), 1)
    dst.update_tags(units='dimensionless', method='chavez-cost-1996')
print("Pixels válidos:", int(np.isfinite(imagem).sum()), "de", imagem.size)
print("Saída:", saida.resolve())

## Como conferir a execução

Abra o GeoTIFF salvo em um SIG e confira a posição, a unidade e a banda informadas acima. Compare a contagem de pixels válidos com a área disponível: QA pode excluir pixels, e os critérios do próprio método podem produzir lacunas adicionais.

| Se acontecer | O que conferir |
| --- | --- |
| `FileNotFoundError` no passo 1 | Preencha os caminhos listados no erro, inclusive MTL quando solicitado e ambas as QA. |
| Erro de grade no passo 2 | Prepare arquivos com a mesma projeção, transformação, dimensões e recorte. Não redimensione as QA por interpolação contínua. |
| Erro de domínio no passo 3 | Confira a grandeza/unidade e os requisitos do método. |
| Saída com NaN | Confira as contagens QA/nodata e o suporte do método; NaN não deve ser substituído por temperatura inventada. |

[Voltar ao índice e à ordem de execução](README.md).


## Consulta — opções e detalhes do método

O roteiro executável termina acima. Leia os tópicos abaixo para adaptar a entrada ou consultar parâmetros; não execute todas as alternativas em sequência sobre a mesma saída.


### Entradas da função e opções

### Entradas do caminho principal

| Entrada da função | O que fornecer | Unidade / domínio |
|---|---|---|
| `toa_reflectance` | Imagem óptica em reflectância TOA, já ajustada pelo ângulo solar | Adimensional, [0,1] |

**Além da imagem:** `sun_elevation`, a elevação solar real da cena em graus, em (0,90], obtida do MTL. Não é zênite e não há valor assumido neste exemplo. O ajuste exige pelo menos dois pixels válidos.

Se já possui o objeto `parameters` de um ajuste deste método, reutilize-o em vez de informar elevação solar e opções de ajuste novamente; veja a consulta de alternativas. A imagem de entrada continua sendo reflectância TOA.

Todas as imagens usadas juntas devem ter a mesma grade: projeção, resolução, posição dos pixels e dimensões. As variáveis da chamada são arrays de pixels; não são caminhos de arquivo.

### O que é opcional

- **`mask`:** máscara booleana alinhada: True exclui o pixel e False permite avaliar. Opcional na API; neste exemplo é preenchida no passo 2 com QA e nodata. Não passar QA_PIXEL bruta.
- **`percentile` e `assumed_dark_surface`:** seleção e reflectância do alvo escuro; ambos usam 0,01 por padrão, com significados diferentes.
- **`return_diagnostics`:** permite obter o objeto parameters para reutilizar o ajuste.

### Alternativas de uso

Escolha a alternativa que corresponde aos dados disponíveis; não execute todas as chamadas em sequência sobre a mesma saída.

### 5.1. Reutilizar um ajuste pronto

Fazer o ajuste uma vez na área completa de calibração. Na reutilização, escolher o mesmo método e fornecer parameters; omitir sun_elevation, percentile e assumed_dark_surface. Não ajustar separadamente em cada bloco.

```python
ajuste = optical_correction(
    method='chavez-cost-1996',
    toa_reflectance=rho_toa,
    sun_elevation=elevacao_solar,
    mask=mascara,
    return_diagnostics=True,
)

parametros = ajuste["parameters"]
imagem = optical_correction(
    method='chavez-cost-1996',
    toa_reflectance=rho_toa,
    parameters=parametros,
    mask=mascara,
)
```
### Diagnósticos

Com return_diagnostics=True, o resultado contém reflectance, parameters, method, units e invalid_mask. Use resultado["reflectance"] para salvar a imagem e guarde o objeto parameters para outros blocos. Não aplicar a correção novamente sobre uma imagem já corrigida.

### Referência completa dos parâmetros

### Caminho principal

A chamada completa e a tabela abaixo seguem a mesma ordem.

```python
resultado = optical_correction(
    method='chavez-cost-1996',
    toa_reflectance=rho_toa,
    sun_elevation=elevacao_solar,
    percentile=.01,
    assumed_dark_surface=.01,
    mask=mascara,
    return_diagnostics=diagnosticos,
)
```

| Parâmetro | Condição | Padrão da API | Unidade | Detalhamento e opções |
|---|---|---|---|---|
| `method` | Obrigatório neste exemplo: selecionar o método | `"chavez-cost-1996"` | Seletor | Usar "chavez-cost-1996" neste notebook. Os outros métodos têm requisitos próprios e notebooks separados. |
| `toa_reflectance` | Obrigatório | `Sem padrão` | Reflectância TOA, [0,1] | Imagem já corrigida pelo ângulo solar. Não fornecer DN, radiância ou reflectância de superfície existente. |
| `sun_elevation` | Obrigatório para ajustar; omitir com parameters | `None` | Graus, (0,90] | Um único ângulo de elevação solar para a cena, obtido do MTL. Não é ângulo zenital. |
| `percentile` | Opcional no ajuste; omitir na reutilização | `None → 0.01` | Percentil, [0,100] | 0.01 significa percentil 0,01, não 1%. Seleção operacional do objeto escuro; ajustar na cena completa válida. |
| `assumed_dark_surface` | Opcional no ajuste; omitir na reutilização | `None → 0.01` | Reflectância, [0,1] | Reflectância assumida do alvo escuro; deve ser compatível com o percentil observado. Não há recorte silencioso de haze negativo. |
| `mask` | Opcional na API; fornecido neste exemplo | `None` | Booleano | Máscara de exclusão alinhada: True rejeita, False permite avaliar. O passo 2 lê ambas as QA e combina seus bits com nodata. Água segue a escolha exclude_water explícita neste notebook. Não fornecer QA bruta; a função pode acrescentar rejeições por domínio. |
| `return_diagnostics` | Opcional | `False` | Booleano | False retorna somente a imagem; True retorna dicionário com: reflectance, parameters, method, units, invalid_mask. A máscara final pode incluir rejeições adicionais à máscara de entrada. |

### Parâmetros das alternativas

As entradas substituídas e as combinações permitidas são explicadas na consulta de alternativas. Os demais parâmetros permanecem os da tabela principal.

| Parâmetro | Condição | Padrão da API | Unidade | Detalhamento e opções |
|---|---|---|---|---|
| `parameters` | Obrigatório na reutilização de ajuste | `None` | Objeto de parâmetros | Usar resultado["parameters"] obtido com return_diagnostics=True no mesmo método. Omitir sun_elevation, percentile e assumed_dark_surface nessa rota; ajustar uma vez na cena e reutilizar nos blocos. |

Os valores escolhidos no exemplo podem diferir do padrão da API; a coluna de padrão descreve a função. As máscaras e as validações de domínio podem produzir NaN, sem preenchimento automático.

### Aplicabilidade e limites

Ajustar na área completa de calibração e reutilizar os mesmos parâmetros nos blocos. percentile=0,01 significa percentil 0,01, não 1%; é uma adaptação computacional, não um percentil universal do artigo. Haze negativo gera erro; resultados corrigidos fora de [0,1] viram NaN. Transferência TM→OLI/OLI-2 é experimental; DOS/COST não equivale a LaSRC, SR operacional ou correção atmosférica térmica.

Execução e consistência numérica não demonstram acurácia independente em campo. A parametrização deve ser adequada à banda, sensor, superfície e condições da aplicação.

Veja a [ficha metodologica](../docs/method-cards.md) para equacao, coeficientes, dominio e adaptacoes. Execucao numericamente correta nao estabelece acuracia de campo. [Voltar ao indice](README.md).